# Robustness Runner MTF Microstructure — Revised

Notebook ini adalah versi revisi yang **self-contained** dan siap dijalankan untuk:
- load data M1
- QC + gap map
- resample ke **M5 / M15 / H1**
- feature engineering
- merge context H1 ke M15
- optional **M5 confirmation**
- generate event families
- run baseline backtest
- run robustness battery:
  - family isolation
  - session isolation
  - parameter grid
  - adverse execution
  - gap sensitivity
  - leave-one-year-out style
  - event combinations
  - ATR floor/cap
  - Monte Carlo
- export semua tabel ke CSV

Semua rule dibuat **backward-looking** dan **SL-first on intrabar tie**.


In [ ]:

import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


In [ ]:

# =========================
# User Config
# =========================
CONFIG = {
    # ganti path ini sesuai file CSV kamu
    "data_path": "../../data/raw/XAUUSD_M1.csv",
    "output_dir": "outputs_robustness_runner_revised",

    "source_tf": "M1",
    "execution_tf": "M15",
    "signal_tf": "M15",
    "context_tf": "H1",
    "refine_tf": "M5",

    "resample_label": "right",
    "resample_closed": "right",

    "spread_points": 0.0,
    "slippage_points": 0.0,
    "entry_delay_bars": 0,                  # 0 = next bar open

    "tp_r_multiple": 2.0,
    "timeout_bars": 16,
    "rolling_level_n": 20,
    "swing_lookback": 3,
    "cooldown_bars": 0,

    "train_ratio": 0.6,
    "val_ratio": 0.2,

    "position_policy": "one_at_a_time",

    "enabled_event_families": [
        "sweep_reclaim",
        "rejection",
        "compression_expansion",
        "momentum_break",
    ],
    "enabled_sessions": ["asia", "london", "ny_overlap", "ny"],

    "atr_floor_mult": None,
    "atr_cap_mult": None,

    "gap_guard_minutes": 5,

    # Optional M5 refinement
    "use_m5_confirmation": False,
    "m5_confirmation_window_bars": 3,
    "m5_confirmation_mode": "simple_reclaim",   # simple_reclaim | micro_break
}
OUTDIR = Path(CONFIG["output_dir"])
OUTDIR.mkdir(parents=True, exist_ok=True)
CONFIG


In [ ]:

# =========================
# IO & Data Contract
# =========================
def load_ohlcv(path: str) -> pd.DataFrame:
    names = ['date', 'time', 'open', 'high', 'low', 'close', 'volume']
    df = pd.read_csv(path, names=names, skiprows=1)
    ts = pd.to_datetime(df['date'] + ' ' + df['time'], format='%Y.%m.%d %H:%M', utc=True)
    df = df.drop(columns=['date', 'time'])
    df.index = ts
    df.index.name = 'timestamp'
    df = df.sort_index()
    return df

def qc_report(df_m1: pd.DataFrame) -> dict:
    idx = df_m1.index
    dup = int(idx.duplicated().sum())
    nonpositive = int(((df_m1[['open','high','low','close']] <= 0).any(axis=1)).sum())
    null_rows = int(df_m1.isnull().any(axis=1).sum())
    missing = int(((idx.to_series().diff().dropna() / pd.Timedelta(minutes=1)) - 1).clip(lower=0).sum())
    return {
        "rows": int(len(df_m1)),
        "start": str(df_m1.index.min()),
        "end": str(df_m1.index.max()),
        "duplicate_index": dup,
        "missing_1m_gaps": missing,
        "nonpositive_price_rows": nonpositive,
        "null_rows": null_rows,
    }

def resample_ohlcv(df: pd.DataFrame, rule: str, label='right', closed='right') -> pd.DataFrame:
    agg = {
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }
    return df.resample(rule, label=label, closed=closed).agg(agg).dropna()

df_m1 = load_ohlcv(CONFIG["data_path"])
qc = qc_report(df_m1)
with open(OUTDIR / "qc_report.json", "w") as f:
    json.dump(qc, f, indent=2)

qc


In [ ]:

# =========================
# Missing-gap map for sensitivity filters
# =========================
def build_gap_windows(df_m1: pd.DataFrame):
    diffs = df_m1.index.to_series().diff()
    gap_starts = diffs[diffs > pd.Timedelta(minutes=1)].index
    windows = []
    for gap_end in gap_starts:
        prev_loc = df_m1.index.get_loc(gap_end) - 1
        if prev_loc >= 0:
            prev_ts = df_m1.index[prev_loc]
            windows.append((prev_ts, gap_end))
    return windows

GAP_WINDOWS = build_gap_windows(df_m1)
len(GAP_WINDOWS), GAP_WINDOWS[:3]


In [ ]:

# =========================
# Resample to M5 / M15 / H1
# =========================
df_m5  = resample_ohlcv(df_m1, '5min',  label=CONFIG["resample_label"], closed=CONFIG["resample_closed"])
df_m15 = resample_ohlcv(df_m1, '15min', label=CONFIG["resample_label"], closed=CONFIG["resample_closed"])
df_h1  = resample_ohlcv(df_m1, '1h',    label=CONFIG["resample_label"], closed=CONFIG["resample_closed"])

print("M1 rows:", len(df_m1), "M5 rows:", len(df_m5), "M15 rows:", len(df_m15), "H1 rows:", len(df_h1))
display(df_m5.head(3))
display(df_m15.head(3))
display(df_h1.head(3))


In [ ]:

# =========================
# Feature Library
# =========================
def add_common_features(df: pd.DataFrame, rolling_n: int = 20) -> pd.DataFrame:
    out = df.copy()
    out['range'] = out['high'] - out['low']
    out['body'] = out['close'] - out['open']
    out['abs_body'] = out['body'].abs()
    out['upper_wick'] = out['high'] - out[['open','close']].max(axis=1)
    out['lower_wick'] = out[['open','close']].min(axis=1) - out['low']
    out['body_ratio'] = np.where(out['range'] > 0, out['abs_body'] / out['range'], 0.0)
    out['close_pos'] = np.where(out['range'] > 0, (out['close'] - out['low']) / out['range'], 0.5)

    for n in [1, 3, 5, 10]:
        out[f'ret_{n}'] = out['close'].pct_change(n)

    out['ema20'] = out['close'].ewm(span=20, adjust=False).mean()
    out['ema50'] = out['close'].ewm(span=50, adjust=False).mean()
    out['ema20_slope'] = out['ema20'].diff()
    out['ema50_slope'] = out['ema50'].diff()
    out['dist_ema20'] = out['close'] - out['ema20']
    out['dist_ema50'] = out['close'] - out['ema50']

    out['bb_mid'] = out['close'].rolling(20).mean()
    out['bb_std'] = out['close'].rolling(20).std(ddof=0)
    out['bb_upper'] = out['bb_mid'] + 2 * out['bb_std']
    out['bb_lower'] = out['bb_mid'] - 2 * out['bb_std']
    out['bb_width'] = out['bb_upper'] - out['bb_lower']

    prev_close = out['close'].shift(1)
    tr = pd.concat([
        out['high'] - out['low'],
        (out['high'] - prev_close).abs(),
        (out['low'] - prev_close).abs()
    ], axis=1).max(axis=1)
    out['atr14'] = tr.rolling(14).mean()

    out['vol_5'] = out['ret_1'].rolling(5).std(ddof=0)
    out['vol_10'] = out['ret_1'].rolling(10).std(ddof=0)
    out['vol_20'] = out['ret_1'].rolling(20).std(ddof=0)

    out[f'rolling_high_{rolling_n}'] = out['high'].shift(1).rolling(rolling_n).max()
    out[f'rolling_low_{rolling_n}'] = out['low'].shift(1).rolling(rolling_n).min()

    out['range_mean_10'] = out['range'].rolling(10).mean()
    out['compression_ratio'] = np.where(out['range_mean_10'] > 0, out['range'] / out['range_mean_10'], np.nan)

    return out

df_m5f = add_common_features(df_m5, CONFIG["rolling_level_n"])
df_m15f = add_common_features(df_m15, CONFIG["rolling_level_n"])
df_h1f = add_common_features(df_h1, CONFIG["rolling_level_n"])

display(df_m15f.head(3))


In [ ]:

# =========================
# Merge H1 context into M15, and M5 refinement context into M15
# =========================
def add_h1_context_to_m15(df_m15f: pd.DataFrame, df_h1f: pd.DataFrame) -> pd.DataFrame:
    h1_cols = ['ema20','ema50','ema20_slope','ema50_slope','bb_width','atr14','close']
    h1 = df_h1f[h1_cols].copy().add_prefix('h1_')
    merged = pd.merge_asof(
        df_m15f.sort_index(),
        h1.sort_index(),
        left_index=True,
        right_index=True,
        direction='backward',
        allow_exact_matches=True
    )
    return merged

df = add_h1_context_to_m15(df_m15f, df_h1f)
df.head(3)


In [ ]:

# =========================
# Session tagging
# =========================
def session_name(ts: pd.Timestamp) -> str:
    h = ts.hour
    if 0 <= h < 7:
        return 'asia'
    elif 7 <= h < 12:
        return 'london'
    elif 12 <= h < 17:
        return 'ny_overlap'
    else:
        return 'ny'

df['session'] = [session_name(ts) for ts in df.index]
df['year'] = df.index.year
df[['session','year']].head()


In [ ]:

# =========================
# Context regime
# =========================
def compute_context_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['ctx_up'] = (out['h1_ema20'] > out['h1_ema50']) & (out['h1_ema20_slope'] > 0)
    out['ctx_dn'] = (out['h1_ema20'] < out['h1_ema50']) & (out['h1_ema20_slope'] < 0)
    out['ctx_bb_wide'] = out['h1_bb_width'] > out['h1_bb_width'].rolling(100).median()
    out['ctx_vol_high'] = out['h1_atr14'] > out['h1_atr14'].rolling(100).median()
    return out

df = compute_context_flags(df)
df[['ctx_up','ctx_dn','ctx_bb_wide','ctx_vol_high']].head()


In [ ]:

# =========================
# Backward-only swing stop helpers
# =========================
def structural_long_stop(df: pd.DataFrame, idx: pd.Index, lookback: int) -> pd.Series:
    return df['low'].shift(1).rolling(lookback).min().reindex(idx)

def structural_short_stop(df: pd.DataFrame, idx: pd.Index, lookback: int) -> pd.Series:
    return df['high'].shift(1).rolling(lookback).max().reindex(idx)

def apply_atr_floor_cap(entry_price, stop_price, atr, side, atr_floor_mult=None, atr_cap_mult=None):
    if pd.isna(stop_price) or pd.isna(entry_price) or pd.isna(atr) or atr <= 0:
        return stop_price
    risk = abs(entry_price - stop_price)
    min_risk = atr_floor_mult * atr if atr_floor_mult is not None else None
    max_risk = atr_cap_mult * atr if atr_cap_mult is not None else None
    adj_risk = risk
    if min_risk is not None:
        adj_risk = max(adj_risk, min_risk)
    if max_risk is not None:
        adj_risk = min(adj_risk, max_risk)
    if side == 'long':
        return entry_price - adj_risk
    return entry_price + adj_risk


In [ ]:

# =========================
# Event-family generators
# =========================
RLN = CONFIG["rolling_level_n"]

def generate_sweep_reclaim(df: pd.DataFrame):
    out = []
    cond_long = (
        (df['high'] > df[f'rolling_high_{RLN}']) &
        (df['close'] < df[f'rolling_high_{RLN}']) &
        df['ctx_up']
    )
    cond_short = (
        (df['low'] < df[f'rolling_low_{RLN}']) &
        (df['close'] > df[f'rolling_low_{RLN}']) &
        df['ctx_dn']
    )
    for ts in df.index[cond_long.fillna(False)]:
        out.append({"event_time": ts, "side": "long", "event_family": "sweep_reclaim"})
    for ts in df.index[cond_short.fillna(False)]:
        out.append({"event_time": ts, "side": "short", "event_family": "sweep_reclaim"})
    return pd.DataFrame(out)

def generate_rejection(df: pd.DataFrame):
    out = []
    cond_long = (
        (df['lower_wick'] > df['abs_body'] * 1.5) &
        (df['close_pos'] > 0.6) &
        df['ctx_up']
    )
    cond_short = (
        (df['upper_wick'] > df['abs_body'] * 1.5) &
        (df['close_pos'] < 0.4) &
        df['ctx_dn']
    )
    for ts in df.index[cond_long.fillna(False)]:
        out.append({"event_time": ts, "side": "long", "event_family": "rejection"})
    for ts in df.index[cond_short.fillna(False)]:
        out.append({"event_time": ts, "side": "short", "event_family": "rejection"})
    return pd.DataFrame(out)

def generate_compression_expansion(df: pd.DataFrame):
    out = []
    med_comp = df['compression_ratio'].rolling(100).median()
    cond_long = (
        (df['compression_ratio'] > 1.8) &
        (df['compression_ratio'].shift(1) < med_comp.shift(1)) &
        (df['close'] > df['open']) &
        df['ctx_up']
    )
    cond_short = (
        (df['compression_ratio'] > 1.8) &
        (df['compression_ratio'].shift(1) < med_comp.shift(1)) &
        (df['close'] < df['open']) &
        df['ctx_dn']
    )
    for ts in df.index[cond_long.fillna(False)]:
        out.append({"event_time": ts, "side": "long", "event_family": "compression_expansion"})
    for ts in df.index[cond_short.fillna(False)]:
        out.append({"event_time": ts, "side": "short", "event_family": "compression_expansion"})
    return pd.DataFrame(out)

def generate_momentum_break(df: pd.DataFrame):
    out = []
    cond_long = (
        (df['body_ratio'] > 0.65) &
        (df['close'] > df['high'].shift(1)) &
        df['ctx_up']
    )
    cond_short = (
        (df['body_ratio'] > 0.65) &
        (df['close'] < df['low'].shift(1)) &
        df['ctx_dn']
    )
    for ts in df.index[cond_long.fillna(False)]:
        out.append({"event_time": ts, "side": "long", "event_family": "momentum_break"})
    for ts in df.index[cond_short.fillna(False)]:
        out.append({"event_time": ts, "side": "short", "event_family": "momentum_break"})
    return pd.DataFrame(out)

event_frames = [
    generate_sweep_reclaim(df),
    generate_rejection(df),
    generate_compression_expansion(df),
    generate_momentum_break(df),
]
events = pd.concat([e for e in event_frames if len(e)], ignore_index=True).sort_values("event_time").reset_index(drop=True)
events.head(), events['event_family'].value_counts()


In [ ]:

# =========================
# Gap guard / split helpers
# =========================
def event_near_gap(ts: pd.Timestamp, gap_windows, guard_minutes=5) -> bool:
    guard = pd.Timedelta(minutes=guard_minutes)
    for a, b in gap_windows:
        if (a - guard) <= ts <= (b + guard):
            return True
    return False

def assign_time_split(index: pd.Index, train_ratio=0.6, val_ratio=0.2) -> pd.Series:
    n = len(index)
    a = int(n * train_ratio)
    b = int(n * (train_ratio + val_ratio))
    labels = np.array(['test'] * n, dtype=object)
    labels[:a] = 'train'
    labels[a:b] = 'validation'
    return pd.Series(labels, index=index)

events['near_gap'] = events['event_time'].apply(lambda x: event_near_gap(x, GAP_WINDOWS, CONFIG["gap_guard_minutes"]))
events['session'] = events['event_time'].map(df['session'])
events['year'] = events['event_time'].map(df['year'])

bar_split = assign_time_split(df.index, CONFIG["train_ratio"], CONFIG["val_ratio"])
events['split'] = events['event_time'].map(bar_split)

events.head(), events['split'].value_counts()


In [ ]:

# =========================
# Optional M5 confirmation
# =========================
def check_m5_confirmation(event_time: pd.Timestamp, side: str, df_m5f: pd.DataFrame, config: dict) -> bool:
    if not config.get("use_m5_confirmation", False):
        return True

    # use only M5 bars strictly after event_time
    future = df_m5f.loc[df_m5f.index > event_time].head(int(config.get("m5_confirmation_window_bars", 3)))
    if len(future) == 0:
        return False

    mode = config.get("m5_confirmation_mode", "simple_reclaim")

    if mode == "simple_reclaim":
        if side == "long":
            return bool((future['close'] > future['open']).any())
        else:
            return bool((future['close'] < future['open']).any())

    if mode == "micro_break":
        if side == "long":
            return bool((future['close'] > future['high'].shift(1)).fillna(False).any())
        else:
            return bool((future['close'] < future['low'].shift(1)).fillna(False).any())

    return True


In [ ]:

# =========================
# Backtest Engine
# =========================
def run_backtest(df: pd.DataFrame, events: pd.DataFrame, config: dict, df_m5f: pd.DataFrame = None) -> pd.DataFrame:
    events = events.copy()
    if len(events) == 0:
        return pd.DataFrame()

    if config.get("enabled_event_families"):
        events = events[events['event_family'].isin(config["enabled_event_families"])].copy()

    if config.get("enabled_sessions"):
        events = events[events['session'].isin(config["enabled_sessions"])].copy()

    if config.get("exclude_near_gap", False):
        events = events[~events['near_gap']].copy()

    if config.get("year_filter") is not None:
        events = events[events['year'].isin(config["year_filter"])].copy()

    if len(events) == 0:
        return pd.DataFrame()

    events = events.sort_values('event_time').reset_index(drop=True)
    index = df.index
    trades = []
    last_exit_loc = -1

    rolling_long = df['low'].shift(1).rolling(config["swing_lookback"]).min()
    rolling_short = df['high'].shift(1).rolling(config["swing_lookback"]).max()

    for _, ev in events.iterrows():
        et = ev['event_time']

        if df_m5f is not None and config.get("use_m5_confirmation", False):
            if not check_m5_confirmation(et, ev['side'], df_m5f, config):
                continue

        try:
            event_loc = index.get_loc(et)
        except KeyError:
            continue

        entry_loc = event_loc + 1 + int(config.get("entry_delay_bars", 0))
        if entry_loc >= len(index):
            continue

        # one-at-a-time policy
        if config.get("position_policy", "one_at_a_time") == "one_at_a_time" and entry_loc <= last_exit_loc:
            continue

        entry_time = index[entry_loc]
        entry_bar = df.iloc[entry_loc]
        entry_price = float(entry_bar['open'])

        if ev['side'] == 'long':
            stop = float(rolling_long.iloc[entry_loc]) if not pd.isna(rolling_long.iloc[entry_loc]) else np.nan
        else:
            stop = float(rolling_short.iloc[entry_loc]) if not pd.isna(rolling_short.iloc[entry_loc]) else np.nan

        atr = float(df['atr14'].iloc[entry_loc]) if not pd.isna(df['atr14'].iloc[entry_loc]) else np.nan
        stop = apply_atr_floor_cap(
            entry_price=entry_price,
            stop_price=stop,
            atr=atr,
            side=ev['side'],
            atr_floor_mult=config.get("atr_floor_mult"),
            atr_cap_mult=config.get("atr_cap_mult")
        )

        if pd.isna(stop):
            continue

        risk = abs(entry_price - stop)
        if risk <= 0:
            continue

        half_spread = float(config.get("spread_points", 0.0)) / 2.0
        slip = float(config.get("slippage_points", 0.0))

        if ev['side'] == 'long':
            fill_entry = entry_price + half_spread + slip
            tp = fill_entry + risk * config["tp_r_multiple"]
            sl = stop
        else:
            fill_entry = entry_price - half_spread - slip
            tp = fill_entry - risk * config["tp_r_multiple"]
            sl = stop

        timeout_bars = int(config["timeout_bars"])
        max_exit_loc = min(entry_loc + timeout_bars, len(index) - 1)

        exit_loc = max_exit_loc
        exit_time = index[exit_loc]
        exit_price = float(df['close'].iloc[exit_loc])
        exit_reason = 'timeout'
        r_mult = (exit_price - fill_entry) / risk if ev['side'] == 'long' else (fill_entry - exit_price) / risk

        for bar_loc in range(entry_loc, max_exit_loc + 1):
            bar = df.iloc[bar_loc]
            hi = float(bar['high'])
            lo = float(bar['low'])

            if ev['side'] == 'long':
                hit_sl = lo <= sl
                hit_tp = hi >= tp
                if hit_sl and hit_tp:
                    # conservative tie-break: SL first
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = sl
                    exit_reason = 'sl'
                    r_mult = -1.0
                    break
                elif hit_sl:
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = sl
                    exit_reason = 'sl'
                    r_mult = -1.0
                    break
                elif hit_tp:
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = tp
                    exit_reason = 'tp'
                    r_mult = config["tp_r_multiple"]
                    break
            else:
                hit_sl = hi >= sl
                hit_tp = lo <= tp
                if hit_sl and hit_tp:
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = sl
                    exit_reason = 'sl'
                    r_mult = -1.0
                    break
                elif hit_sl:
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = sl
                    exit_reason = 'sl'
                    r_mult = -1.0
                    break
                elif hit_tp:
                    exit_loc = bar_loc
                    exit_time = index[bar_loc]
                    exit_price = tp
                    exit_reason = 'tp'
                    r_mult = config["tp_r_multiple"]
                    break

        trades.append({
            "event_time": et,
            "entry_time": entry_time,
            "exit_time": exit_time,
            "event_name": ev['event_family'],
            "side": ev['side'],
            "session": ev['session'],
            "year": ev['year'],
            "split": ev['split'],
            "entry_price": fill_entry,
            "stop_price": sl,
            "take_profit": tp,
            "exit_price": exit_price,
            "exit_reason": exit_reason,
            "risk": risk,
            "r": float(r_mult),
            "holding_bars": int(exit_loc - entry_loc),
            "near_gap": bool(ev['near_gap']),
        })

        if config.get("position_policy", "one_at_a_time") == "one_at_a_time":
            last_exit_loc = exit_loc + int(config.get("cooldown_bars", 0))

    return pd.DataFrame(trades)


In [ ]:

# =========================
# Metrics & summaries
# =========================
def max_drawdown_r(r: pd.Series) -> float:
    eq = r.cumsum()
    peak = eq.cummax()
    dd = eq - peak
    return float(dd.min()) if len(dd) else 0.0

def max_losing_streak(r: pd.Series) -> int:
    streak = best = 0
    for x in r:
        if x < 0:
            streak += 1
            best = max(best, streak)
        else:
            streak = 0
    return int(best)

def compute_metrics(trades: pd.DataFrame) -> dict:
    if len(trades) == 0:
        return {
            "total_trades": 0,
            "win_rate": 0.0,
            "profit_factor": 0.0,
            "expectancy_r": 0.0,
            "avg_r": 0.0,
            "median_r": 0.0,
            "sum_r": 0.0,
            "max_drawdown_r": 0.0,
            "max_losing_streak": 0,
            "avg_holding_bars": 0.0,
        }
    r = trades['r'].astype(float)
    gp = float(r[r > 0].sum())
    gl = float(-r[r < 0].sum())
    pf = gp / gl if gl > 0 else np.inf
    return {
        "total_trades": int(len(trades)),
        "win_rate": float((r > 0).mean()),
        "profit_factor": float(pf),
        "expectancy_r": float(r.mean()),
        "avg_r": float(r.mean()),
        "median_r": float(r.median()),
        "sum_r": float(r.sum()),
        "max_drawdown_r": float(max_drawdown_r(r)),
        "max_losing_streak": int(max_losing_streak(r)),
        "avg_holding_bars": float(trades['holding_bars'].mean()),
    }

def summarize_by_group(trades: pd.DataFrame, group_col: str) -> pd.DataFrame:
    if len(trades) == 0:
        return pd.DataFrame()
    rows = []
    for key, g in trades.groupby(group_col):
        row = compute_metrics(g)
        row[group_col] = key
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_col).reset_index(drop=True)

def plot_equity(trades: pd.DataFrame, title="Equity Curve in R"):
    if len(trades) == 0:
        print("No trades.")
        return
    tmp = trades.sort_values('exit_time').copy()
    tmp['cum_r'] = tmp['r'].cumsum()
    plt.figure(figsize=(12, 5))
    plt.plot(tmp['exit_time'], tmp['cum_r'])
    plt.title(title)
    plt.xlabel("Exit Time")
    plt.ylabel("Cumulative R")
    plt.grid(True, alpha=0.3)
    plt.show()


In [ ]:

# =========================
# Baseline run
# =========================
baseline_trades = run_backtest(df, events, CONFIG, df_m5f=df_m5f)
baseline_metrics = compute_metrics(baseline_trades)

display(pd.DataFrame([baseline_metrics]))
display(summarize_by_group(baseline_trades, 'split'))
display(summarize_by_group(baseline_trades, 'event_name'))
display(summarize_by_group(baseline_trades, 'session'))
display(summarize_by_group(baseline_trades, 'year'))

plot_equity(baseline_trades, "Baseline Equity Curve in R")


In [ ]:

# =========================
# Runner utility
# =========================
def run_scenario(
    name: str,
    base_config: dict,
    enabled_families=None,
    enabled_sessions=None,
    overrides=None,
    exclude_near_gap=False,
    year_filter=None
):
    cfg = dict(base_config)
    if enabled_families is not None:
        cfg["enabled_event_families"] = enabled_families
    if enabled_sessions is not None:
        cfg["enabled_sessions"] = enabled_sessions
    if overrides:
        cfg.update(overrides)
    cfg["exclude_near_gap"] = bool(exclude_near_gap)
    cfg["year_filter"] = year_filter

    trades = run_backtest(df, events, cfg, df_m5f=df_m5f)
    m = compute_metrics(trades)
    m["scenario"] = name
    if enabled_families is not None:
        m["families"] = ",".join(enabled_families)
    if enabled_sessions is not None:
        m["sessions"] = ",".join(enabled_sessions)
    if overrides:
        for k, v in overrides.items():
            m[k] = v
    if year_filter is not None:
        m["year_filter"] = ",".join(map(str, year_filter))
    m["exclude_near_gap"] = bool(exclude_near_gap)
    return trades, m


In [ ]:

# 1) Family isolation
family_results = []
for fam in ["sweep_reclaim", "rejection", "compression_expansion", "momentum_break"]:
    _, m = run_scenario(name=f"family::{fam}", base_config=CONFIG, enabled_families=[fam])
    family_results.append(m)

family_results_df = pd.DataFrame(family_results).sort_values("profit_factor", ascending=False).reset_index(drop=True)
family_results_df


In [ ]:

# 2) Session isolation
session_specs = {
    "asia_only": ["asia"],
    "london_only": ["london"],
    "ny_overlap_only": ["ny_overlap"],
    "ny_only": ["ny"],
    "london_ny": ["london", "ny_overlap", "ny"],
    "all_sessions": ["asia", "london", "ny_overlap", "ny"],
}
session_results = []
for nm, sess in session_specs.items():
    _, m = run_scenario(name=f"session::{nm}", base_config=CONFIG, enabled_sessions=sess)
    session_results.append(m)

session_results_df = pd.DataFrame(session_results).sort_values("profit_factor", ascending=False).reset_index(drop=True)
session_results_df


In [ ]:

# 3) Parameter perturbation grid
tp_grid = [1.5, 2.0, 2.5]
timeout_grid = [8, 12, 16, 20]
swing_grid = [2, 3, 4, 5]

grid_rows = []
for tp in tp_grid:
    for tout in timeout_grid:
        for sw in swing_grid:
            _, m = run_scenario(
                name=f"grid::tp{tp}_to{tout}_sw{sw}",
                base_config=CONFIG,
                overrides={"tp_r_multiple": tp, "timeout_bars": tout, "swing_lookback": sw}
            )
            grid_rows.append(m)

grid_df = pd.DataFrame(grid_rows).sort_values("profit_factor", ascending=False).reset_index(drop=True)
display(grid_df.head(20))

pivot_pf = grid_df.pivot_table(index="timeout_bars", columns="swing_lookback", values="profit_factor", aggfunc="mean")
pivot_exp = grid_df.pivot_table(index="timeout_bars", columns="swing_lookback", values="expectancy_r", aggfunc="mean")

display(pivot_pf)
display(pivot_exp)

plt.figure(figsize=(6, 4))
plt.imshow(pivot_pf.values, aspect='auto')
plt.xticks(range(len(pivot_pf.columns)), pivot_pf.columns)
plt.yticks(range(len(pivot_pf.index)), pivot_pf.index)
plt.colorbar(label='Profit Factor')
plt.title("Grid PF Plateau")
plt.xlabel("swing_lookback")
plt.ylabel("timeout_bars")
plt.show()


In [ ]:

# 4) Adverse execution tests
adverse_specs = [
    ("base", {"spread_points": 0.0, "slippage_points": 0.0, "entry_delay_bars": 0}),
    ("spread_10", {"spread_points": 0.10, "slippage_points": 0.0, "entry_delay_bars": 0}),
    ("spread_20", {"spread_points": 0.20, "slippage_points": 0.0, "entry_delay_bars": 0}),
    ("slip_10", {"spread_points": 0.0, "slippage_points": 0.10, "entry_delay_bars": 0}),
    ("slip_20", {"spread_points": 0.0, "slippage_points": 0.20, "entry_delay_bars": 0}),
    ("delay_1", {"spread_points": 0.0, "slippage_points": 0.0, "entry_delay_bars": 1}),
]

adverse_rows = []
for nm, ov in adverse_specs:
    _, m = run_scenario(name=f"adverse::{nm}", base_config=CONFIG, overrides=ov)
    adverse_rows.append(m)

adverse_df = pd.DataFrame(adverse_rows).sort_values("profit_factor", ascending=False).reset_index(drop=True)
adverse_df


In [ ]:

# 5) Gap sensitivity
gap_rows = []
for ex in [False, True]:
    _, m = run_scenario(name=f"gap::exclude_{ex}", base_config=CONFIG, exclude_near_gap=ex)
    gap_rows.append(m)

gap_df = pd.DataFrame(gap_rows)
gap_df


In [ ]:

# 6) Leave-one-year-out style inspection
all_years = sorted(df['year'].dropna().unique().tolist())
loyo_rows = []
for y in all_years:
    _, held = run_scenario(name=f"year_only::{y}", base_config=CONFIG, year_filter=[y])
    held["heldout_year"] = y
    loyo_rows.append(held)

loyo_df = pd.DataFrame(loyo_rows).sort_values("heldout_year").reset_index(drop=True)
loyo_df


In [ ]:

# 7) Event-combination attribution
combo_specs = {
    "sweep_only": ["sweep_reclaim"],
    "rejection_only": ["rejection"],
    "compression_only": ["compression_expansion"],
    "momentum_only": ["momentum_break"],
    "sweep_rejection": ["sweep_reclaim", "rejection"],
    "sweep_compression": ["sweep_reclaim", "compression_expansion"],
    "all_minus_momentum": ["sweep_reclaim", "rejection", "compression_expansion"],
    "all_families": ["sweep_reclaim", "rejection", "compression_expansion", "momentum_break"],
}

combo_rows = []
for nm, fams in combo_specs.items():
    _, m = run_scenario(name=f"combo::{nm}", base_config=CONFIG, enabled_families=fams)
    combo_rows.append(m)

combo_df = pd.DataFrame(combo_rows).sort_values("profit_factor", ascending=False).reset_index(drop=True)
combo_df


In [ ]:

# 8) ATR floor/cap robustness
atr_specs = [
    ("raw_structural", {"atr_floor_mult": None, "atr_cap_mult": None}),
    ("floor_0.5", {"atr_floor_mult": 0.5, "atr_cap_mult": None}),
    ("cap_2.5", {"atr_floor_mult": None, "atr_cap_mult": 2.5}),
    ("floor_0.5_cap_2.5", {"atr_floor_mult": 0.5, "atr_cap_mult": 2.5}),
]

atr_rows = []
for nm, ov in atr_specs:
    _, m = run_scenario(name=f"atr::{nm}", base_config=CONFIG, overrides=ov)
    atr_rows.append(m)

atr_df = pd.DataFrame(atr_rows).sort_values("profit_factor", ascending=False).reset_index(drop=True)
atr_df


In [ ]:

# 9) Optional M5 confirmation check
m5_specs = [
    ("m5_off", {"use_m5_confirmation": False}),
    ("m5_simple_reclaim", {"use_m5_confirmation": True, "m5_confirmation_mode": "simple_reclaim", "m5_confirmation_window_bars": 3}),
    ("m5_micro_break", {"use_m5_confirmation": True, "m5_confirmation_mode": "micro_break", "m5_confirmation_window_bars": 3}),
]
m5_rows = []
for nm, ov in m5_specs:
    _, m = run_scenario(name=f"m5::{nm}", base_config=CONFIG, overrides=ov)
    m5_rows.append(m)

m5_df = pd.DataFrame(m5_rows).sort_values("profit_factor", ascending=False).reset_index(drop=True)
m5_df


In [ ]:

# =========================
# Monte Carlo trade-order stress
# =========================
def monte_carlo_trade_order(trades: pd.DataFrame, n_iter=2000, seed=42):
    if len(trades) == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    r = trades['r'].to_numpy(dtype=float)
    rows = []
    for i in range(n_iter):
        rs = rng.permutation(r)
        eq = np.cumsum(rs)
        dd = np.min(eq - np.maximum.accumulate(eq))
        rows.append({
            "iter": i,
            "sum_r": float(rs.sum()),
            "max_drawdown_r": float(dd),
            "win_rate": float((rs > 0).mean()),
        })
    return pd.DataFrame(rows)

mc_df = monte_carlo_trade_order(baseline_trades, n_iter=2000, seed=42)
mc_df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])


In [ ]:

# =========================
# Leakage & logic sanity checks
# =========================
sanity = {}

sample_loc = min(200, len(df)-1)
sample_ts = df.index[sample_loc]
sanity["sample_context_timestamp"] = str(sample_ts)
valid_h1 = df_h1f.index[df_h1f.index <= sample_ts]
sanity["h1_context_ts_used"] = str(valid_h1[-1]) if len(valid_h1) else None

check_loc = min(100, len(df)-1)
check_ts = df.index[check_loc]
sanity["swing_check_ts"] = str(check_ts)
sanity["prior_lows_used_for_stop"] = df['low'].shift(1).rolling(CONFIG["swing_lookback"]).apply(lambda x: float(np.min(x)), raw=True).iloc[check_loc]
sanity["current_bar_low"] = float(df['low'].iloc[check_loc])
sanity["current_bar_high"] = float(df['high'].iloc[check_loc])

with open(OUTDIR / "sanity_checks.json", "w") as f:
    json.dump(sanity, f, indent=2)

sanity


In [ ]:

# =========================
# Consolidated exports
# =========================
robust_tables = {
    "baseline": pd.DataFrame([baseline_metrics]),
    "family_isolation": family_results_df,
    "session_isolation": session_results_df,
    "parameter_grid": grid_df,
    "adverse_execution": adverse_df,
    "gap_sensitivity": gap_df,
    "year_isolation": loyo_df,
    "event_combinations": combo_df,
    "atr_floor_cap": atr_df,
    "m5_confirmation": m5_df,
    "monte_carlo": mc_df,
}

for name, tbl in robust_tables.items():
    tbl.to_csv(OUTDIR / f"{name}.csv", index=False)

baseline_trades.to_csv(OUTDIR / "baseline_trades.csv", index=False)

with open(OUTDIR / "metrics.json", "w") as f:
    json.dump({"overall_metrics": baseline_metrics}, f, indent=2)

with open(OUTDIR / "run_manifest.json", "w") as f:
    json.dump({"config": CONFIG}, f, indent=2)

print("Exported files to:", OUTDIR.resolve())
sorted([p.name for p in OUTDIR.iterdir()])[:20]


In [ ]:

# Optional: quick pass/fail checklist
def pass_fail_summary(baseline_metrics, adverse_df, grid_df, combo_df, loyo_df,
                      pf_min=1.10, adverse_pf_min=1.05):
    out = {}
    out["baseline_pf_ge_1.10"] = bool(baseline_metrics["profit_factor"] >= pf_min)
    out["baseline_expectancy_positive"] = bool(baseline_metrics["expectancy_r"] > 0)
    out["adverse_min_pf_ge_1.05"] = bool(adverse_df["profit_factor"].min() >= adverse_pf_min) if len(adverse_df) else False
    out["grid_median_pf_ge_1.05"] = bool(grid_df["profit_factor"].median() >= 1.05) if len(grid_df) else False
    out["best_combo_pf_ge_1.10"] = bool(combo_df["profit_factor"].max() >= pf_min) if len(combo_df) else False
    out["yearly_all_positive_expectancy"] = bool((loyo_df["expectancy_r"] > 0).all()) if len(loyo_df) else False
    return out

pf_summary = pass_fail_summary(baseline_metrics, adverse_df, grid_df, combo_df, loyo_df)
pf_summary
